# Notebook 2: Data Cleaning
## Mutual Fund Analytics Platform

This notebook handles data cleaning, validation, and transformation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set paths
BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

# Create processed directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Data Cleaning Started")
print(f"Raw directory: {RAW_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")

Data Cleaning Started
Raw directory: c:\Users\dell\OneDrive\Desktop\bluestock_mf_capstone\data\raw
Processed directory: c:\Users\dell\OneDrive\Desktop\bluestock_mf_capstone\data\processed


In [2]:
# Load raw NAV data
nav_file = RAW_DIR / 'nav_history.csv'
if nav_file.exists():
    df_nav = pd.read_csv(nav_file)
    print(f"Loaded NAV data: {len(df_nav)} rows")
    print(f"Columns: {list(df_nav.columns)}")
else:
    print("NAV file not found")
    df_nav = None

Loaded NAV data: 3915 rows
Columns: ['amfi_code', 'scheme_name', 'nav', 'date']


In [3]:
# Clean NAV data
if df_nav is not None:
    print("\nCleaning NAV data...")
    
    # Convert date column
    df_nav['date'] = pd.to_datetime(df_nav['date'])
    print(f"✓ Date range: {df_nav['date'].min()} to {df_nav['date'].max()}")
    
    # Check for missing values
    missing_before = df_nav.isnull().sum().sum()
    print(f"✓ Missing values before: {missing_before}")
    
    # Remove duplicates
    duplicates = df_nav.duplicated().sum()
    if duplicates > 0:
        df_nav = df_nav.drop_duplicates()
        print(f"✓ Removed {duplicates} duplicate rows")
    
    # Sort by fund and date
    df_nav = df_nav.sort_values(['amfi_code', 'date'])
    
    # Forward fill missing NAV values
    df_nav['nav'] = df_nav.groupby('amfi_code')['nav'].fillna(method='ffill')
    
    # Remove rows with null NAV
    df_nav = df_nav.dropna(subset=['nav'])
    
    # Validate NAV > 0
    df_nav = df_nav[df_nav['nav'] > 0]
    
    # Check missing values after
    missing_after = df_nav.isnull().sum().sum()
    print(f"✓ Missing values after: {missing_after}")
    
    print(f"✓ Cleaned NAV data: {len(df_nav)} rows")
    
    # Save cleaned data
    output_file = PROCESSED_DIR / 'nav_history_cleaned.csv'
    df_nav.to_csv(output_file, index=False)
    print(f"✓ Saved to: {output_file}")


Cleaning NAV data...
✓ Date range: 2023-01-02 00:00:00 to 2025-12-31 00:00:00
✓ Missing values before: 0
✓ Missing values after: 0
✓ Cleaned NAV data: 3915 rows
✓ Saved to: c:\Users\dell\OneDrive\Desktop\bluestock_mf_capstone\data\processed\nav_history_cleaned.csv


In [4]:
# Load and clean transaction data
trans_file = RAW_DIR / 'investor_transactions.csv'
if trans_file.exists():
    df_trans = pd.read_csv(trans_file)
    print(f"Loaded transaction data: {len(df_trans)} rows")
    
    # Clean transaction data
    df_trans['date'] = pd.to_datetime(df_trans['date'])
    df_trans['amount'] = pd.to_numeric(df_trans['amount'], errors='coerce')
    df_trans = df_trans[df_trans['amount'] > 0]
    df_trans = df_trans.drop_duplicates()
    
    # Standardize transaction types
    if 'transaction_type' in df_trans.columns:
        df_trans['transaction_type'] = df_trans['transaction_type'].str.upper()
    
    # Save cleaned data
    output_file = PROCESSED_DIR / 'investor_transactions_cleaned.csv'
    df_trans.to_csv(output_file, index=False)
    print(f"✓ Cleaned transaction data: {len(df_trans)} rows")
    print(f"✓ Saved to: {output_file}")
else:
    print("Transaction file not found")

Loaded transaction data: 5000 rows
✓ Cleaned transaction data: 5000 rows
✓ Saved to: c:\Users\dell\OneDrive\Desktop\bluestock_mf_capstone\data\processed\investor_transactions_cleaned.csv


In [5]:
# Load and clean scheme performance data
perf_file = RAW_DIR / 'scheme_performance.csv'
if perf_file.exists():
    df_perf = pd.read_csv(perf_file)
    print(f"Loaded performance data: {len(df_perf)} rows")
    
    # Clean performance data
    numeric_cols = ['return_1y', 'return_3y', 'return_5y', 'expense_ratio']
    for col in numeric_cols:
        if col in df_perf.columns:
            df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')
    
    # Remove rows with missing critical data
    df_perf = df_perf.dropna(subset=['return_3y'])
    
    # Validate expense ratio range
    if 'expense_ratio' in df_perf.columns:
        df_perf = df_perf[(df_perf['expense_ratio'] >= 0.1) & (df_perf['expense_ratio'] <= 2.5)]
    
    # Save cleaned data
    output_file = PROCESSED_DIR / 'scheme_performance_cleaned.csv'
    df_perf.to_csv(output_file, index=False)
    print(f"✓ Cleaned performance data: {len(df_perf)} rows")
    print(f"✓ Saved to: {output_file}")
else:
    print("Performance file not found")

Loaded performance data: 5 rows
✓ Cleaned performance data: 5 rows
✓ Saved to: c:\Users\dell\OneDrive\Desktop\bluestock_mf_capstone\data\processed\scheme_performance_cleaned.csv


In [6]:
# Data Quality Report
print("\n" + "="*50)
print("DATA QUALITY REPORT")
print("="*50)

if df_nav is not None:
    print(f"\nNAV Data:")
    print(f"  - Total records: {len(df_nav):,}")
    print(f"  - Unique funds: {df_nav['amfi_code'].nunique()}")
    print(f"  - Date range: {df_nav['date'].min()} to {df_nav['date'].max()}")

if 'df_trans' in locals():
    print(f"\nTransaction Data:")
    print(f"  - Total records: {len(df_trans):,}")
    print(f"  - Total amount: ₹{df_trans['amount'].sum():,.2f}")
    print(f"  - Transaction types: {df_trans['transaction_type'].unique().tolist()}")

if 'df_perf' in locals():
    print(f"\nPerformance Data:")
    print(f"  - Total funds: {len(df_perf)}")
    print(f"  - Avg expense ratio: {df_perf['expense_ratio'].mean():.2f}%")


DATA QUALITY REPORT

NAV Data:
  - Total records: 3,915
  - Unique funds: 5
  - Date range: 2023-01-02 00:00:00 to 2025-12-31 00:00:00

Transaction Data:
  - Total records: 5,000
  - Total amount: ₹70,484,000.00
  - Transaction types: ['LUMPSUM', 'SIP', 'REDEMPTION']

Performance Data:
  - Total funds: 5
  - Avg expense ratio: 1.47%


## Data Cleaning Complete

All data has been cleaned and saved to the `data/processed/` directory.

**Cleaned files:**
- `nav_history_cleaned.csv`
- `investor_transactions_cleaned.csv`
- `scheme_performance_cleaned.csv`

Proceed to **Notebook 3: EDA Analysis** for exploratory data analysis.